# PF4 CAD and MI Associations from CARDIoGRAMplusC4D GWAS

This notebook processes coronary artery disease and myocardial infarction summary statistics from the CARDIoGRAMplusC4D datasets to identify SNP associations within the PF4 genomic region.

In [1]:
from pathlib import Path
import json
import pandas as pd

pd.set_option("display.max_columns", None)

In [2]:
region_path = Path("../data/interim/ensembl/region.json")

cad_path = Path("../data/raw/cardiogram_c4d/cardiogram_cad_harmonized.tsv")
mi_path = Path("../data/raw/cardiogram_c4d/cardiogram_mi_harmonized.tsv")

out_dir = Path("../data/interim/cardiogram_c4d")
out_dir.mkdir(parents=True, exist_ok=True)

out_cad_csv = out_dir / "cad_associations.csv"
out_mi_csv = out_dir / "mi_associations.csv"
out_summary_json = out_dir / "summary.json"

In [3]:
with open(region_path, "r") as f:
    region = json.load(f)

chromosome = str(region["chromosome"]).replace("chr", "")
region_start = int(region["region_start"])
region_end = int(region["region_end"])
assembly = region["assembly_name"]

print("Gene:", region["gene_symbol"])
print("Assembly:", assembly)
print("PF4 region:", f"chr{chromosome}:{region_start}-{region_end}")

Gene: PF4
Assembly: GRCh38
PF4 region: chr4:73930811-74032027


In [4]:
for path in [cad_path, mi_path]:
    if not path.exists():
        raise FileNotFoundError(f"Input file not found: {path}")

In [5]:
cardiogram_columns = [
    "hm_variant_id",
    "hm_rsid",
    "hm_chrom",
    "hm_pos",
    "hm_other_allele",
    "hm_effect_allele",
    "hm_beta",
    "hm_odds_ratio",
    "hm_ci_lower",
    "hm_ci_upper",
    "hm_effect_allele_frequency",
    "hm_code",
    "standard_error",
    "p_value",
]

In [6]:
def normalize_chromosome(value):
    value = str(value).strip()
    value = value.replace("chr", "")

    if value.endswith(".0"):
        value = value[:-2]

    return value

In [7]:
def process_cardiogram_dataset(input_path, column_prefix):
    matched_chunks = []

    for chunk in pd.read_csv(
        input_path,
        sep="\t",
        usecols=lambda col: col in cardiogram_columns,
        na_values=["NA", "N/A", ""],
        chunksize=500_000,
        low_memory=False
    ):
        chunk["hm_chrom_filter"] = chunk["hm_chrom"].apply(normalize_chromosome)

        chunk["hm_pos_filter"] = pd.to_numeric(
            chunk["hm_pos"],
            errors="coerce"
        )

        matched = chunk[
            (chunk["hm_chrom_filter"] == chromosome)
            & (chunk["hm_pos_filter"].between(region_start, region_end))
        ].copy()

        matched = matched.drop(
            columns=["hm_chrom_filter", "hm_pos_filter"]
        )

        if not matched.empty:
            matched_chunks.append(matched)

    if matched_chunks:
        associations_df = pd.concat(matched_chunks, ignore_index=True)
    else:
        associations_df = pd.DataFrame(columns=cardiogram_columns)

    rename_map = {
        col: f"{column_prefix}_{col}"
        for col in cardiogram_columns
    }

    associations_df = associations_df.rename(columns=rename_map)

    rows_before_deduplication = len(associations_df)

    associations_df = associations_df.drop_duplicates()

    rows_after_deduplication = len(associations_df)
    duplicate_rows_removed = rows_before_deduplication - rows_after_deduplication

    p_value_col = f"{column_prefix}_p_value"

    if p_value_col in associations_df.columns:
        associations_df[p_value_col] = pd.to_numeric(
            associations_df[p_value_col],
            errors="coerce"
        )

        associations_df = associations_df.sort_values(
            p_value_col,
            ascending=True
        )

    return associations_df, duplicate_rows_removed

In [8]:
cad_associations_df, cad_duplicate_rows_removed = process_cardiogram_dataset(
    input_path=cad_path,
    column_prefix="CAD"
)

cad_associations_df.shape

(192, 14)

In [9]:
mi_associations_df, mi_duplicate_rows_removed = process_cardiogram_dataset(
    input_path=mi_path,
    column_prefix="MI"
)

mi_associations_df.shape

(182, 14)

In [10]:
cad_associations_df.to_csv(out_cad_csv, index=False)
mi_associations_df.to_csv(out_mi_csv, index=False)

print("Saved:", out_cad_csv)
print("Saved:", out_mi_csv)

Saved: ../data/interim/cardiogram_c4d/cad_associations.csv
Saved: ../data/interim/cardiogram_c4d/mi_associations.csv


In [11]:
summary = {
    "source_dataset": "CARDIoGRAMplusC4D harmonized GWAS summary statistics",
    "region": f"chr{chromosome}:{region_start}-{region_end}",
    "region_assembly": assembly,
    "cad": {
        "phenotype": "Coronary artery disease",
        "associations": int(len(cad_associations_df)),
        "unique_rsIDs": int(cad_associations_df["CAD_hm_rsid"].nunique()),
    },
    "mi": {
        "phenotype": "Myocardial infarction",
        "associations": int(len(mi_associations_df)),
        "unique_rsIDs": int(mi_associations_df["MI_hm_rsid"].nunique()),
    },
}

with open(out_summary_json, "w") as f:
    json.dump(summary, f, indent=4)

summary

{'source_dataset': 'CARDIoGRAMplusC4D harmonized GWAS summary statistics',
 'region': 'chr4:73930811-74032027',
 'region_assembly': 'GRCh38',
 'cad': {'phenotype': 'Coronary artery disease',
  'associations': 192,
  'unique_rsIDs': 192},
 'mi': {'phenotype': 'Myocardial infarction',
  'associations': 182,
  'unique_rsIDs': 182}}